# Encoding models

This notebook reproduces Figure 3 of Zada et al. (2025) from the results of `code/encoding.py`. Each feature space, from low-level acoustics to contextual embeddings from GPT-2 XL, was used to fit electrode-wise encoding models at every lag from -2 to +2 seconds around word onset. Here we compare them across the brain, across lags, and within regions of interest.

Run `make encoding` before this notebook.

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from nilearn import datasets
from nilearn.plotting import plot_markers
from statsmodels.stats.multitest import multipletests

from utils import load_results, get_coord_atlas_labels, generate_null_dist, calculate_pvalues

plt.style.use("figures.mplstyle")
Path("../figures/figure3").mkdir(parents=True, exist_ok=True)

## Loading results

Each result file holds a correlation for every electrode and lag, one set per cross-validation fold. We average the two folds and stack the subjects together.

In [ ]:
models = ["spectral", "phonetic", "syntactic", "en.core.web.lg", "gpt2.xl.l24"]

model_results = []
for modelname in models:
    scores, coords, lags = load_results("encoding", modelname)
    print(f"{modelname} scores have a shape of: {scores.shape}")
    model_results.append(scores)

## Brain maps

We summarize each electrode by its best correlation across lags. Electrodes below r = 0.05 are hidden, as in the published panel.

In [ ]:
fig, axes = plt.subplots(5, 1, figsize=(8, 12), dpi=100)

for modelname, scores, ax in zip(models, model_results, axes):
    values = scores.max(-1)
    order = values.argsort()
    plot_markers(values[order], coords[order],
                 node_size=10, display_mode="lzr",
                 node_cmap="magma_r", colorbar=True, alpha=0.8,
                 node_threshold=0.05, node_vmin=0.05, node_vmax=0.25,
                 title=modelname, figure=fig, axes=ax)

fig.savefig("../figures/figure3/model-brainmaps.svg")
plt.show()

## Significance

Each electrode's best correlation is tested against a null distribution of correlations between random vectors, then corrected for multiple comparisons with FDR at p < 0.01. The figure caption describes the electrodes as significant for any feature space, but the paper's code keeps only electrodes significant for all five. We follow the code.

In [ ]:
null_dist = generate_null_dist()  # same size and seed as the paper's code

sig_masks = []
for modelname, scores in zip(models, model_results):
    p_values = calculate_pvalues(scores.max(-1), null_dist)
    sig_mask = multipletests(p_values, alpha=0.01, method="fdr_bh")[0]
    print(f"{modelname}: {sig_mask.sum()} significant electrodes")
    sig_masks.append(sig_mask)

common_mask = np.stack(sig_masks).all(0)
print(f"{common_mask.sum()} electrodes are significant for every model")

## Encoding performance across lags

We average the significant electrodes at each lag for each model. To match the published figure, the error band uses the spread and count of all electrodes, not just the significant ones.

In [ ]:
colors = plt.get_cmap("tab20")(np.arange(5))

fig, ax = plt.subplots(figsize=(4, 3), layout="tight")
ax.axvline(0, c=(.9, .9, .9), ls="--")
ax.axhline(0, c=(.9, .9, .9), ls="--")

for modelname, scores, color in zip(models, model_results, colors):
    mean = scores[common_mask].mean(0)
    err = scores.std(0) / np.sqrt(len(scores))  # all electrodes, as in the paper
    ax.fill_between(lags, mean - err, mean + err, alpha=0.2, color=color)
    ax.plot(lags, mean, color=color, label=modelname, lw=1.8)

ax.set_xlabel("lag (s)")
ax.set_ylabel("encoding performance (r ± sem)")
ax.legend(loc="best")
fig.savefig("../figures/figure3/models-lags.svg")
plt.show()

## Regions of interest

We group Destrieux atlas labels into the six regions shown in the paper. SM and TP here are the paper's PRC and TMP.

In [ ]:
atlas = datasets.fetch_atlas_destrieux_2009()
elec_labels = get_coord_atlas_labels(coords, atlas["maps"], atlas["labels"])

rois = {
    "EAC": ["G_and_S_subcentral", "G_temp_sup-G_T_transv", "S_circular_insula_inf",
            "G_Ins_lg_and_S_cent_ins", "S_temporal_transverse", "Lat_Fis-post", "S_temporal_sup"],
    "STG": ["G_temp_sup-Lateral"],
    "IFG": ["G_front_inf-Triangul", "G_front_inf-Opercular", "G_front_inf-Orbital"],
    "SM": ["G_precentral"],
    "MFG": ["G_front_middle"],
    "TP": ["Pole_temporal", "G_temporal_inf"],
}

First, where the significant electrodes in each region sit:

In [ ]:
fig, axes = plt.subplots(2, 3, sharex=True, sharey=True, dpi=300, figsize=(9, 4))

for (roi, roi_labels), ax in zip(rois.items(), axes.ravel()):
    roi_mask = np.isin(elec_labels, roi_labels) & common_mask
    plot_markers(np.ones(roi_mask.sum()), coords[roi_mask],
                 node_size=10, display_mode="lr",
                 alpha=0.8, colorbar=False,
                 node_cmap="Grays", node_vmin=0, node_vmax=1,
                 figure=fig, axes=ax)
    ax.set_title(f"{roi} ({roi_mask.sum()})", fontsize=8)

fig.savefig("../figures/figure3/brain-schematic-rois.svg")
plt.show()

Then the lag profiles within each region. Like the paper, we leave out the phonetic and syntactic models here to keep the panels readable.

In [ ]:
skip_models = ["phonetic", "syntactic"]

fig, axes = plt.subplots(2, 3, sharex=True, sharey=True, figsize=(4, 3), layout="tight")

for (roi, roi_labels), ax in zip(rois.items(), axes.ravel()):
    roi_mask = np.isin(elec_labels, roi_labels) & common_mask

    ax.axvline(0, c=(.9, .9, .9), ls="--")
    ax.axhline(0, c=(.9, .9, .9), ls="--")
    ax.text(.01, 1, f"{roi}\nN={roi_mask.sum()}", transform=ax.transAxes,
            verticalalignment="top", horizontalalignment="left", fontsize=6)

    for modelname, scores, color in zip(models, model_results, colors):
        if modelname in skip_models:
            continue
        mean = scores[roi_mask].mean(0)
        err = scores.std(0) / np.sqrt(len(scores))  # all electrodes, as in the paper
        ax.fill_between(lags, mean - err, mean + err, alpha=0.2, color=color)
        ax.plot(lags, mean, label=modelname, color=color)

    ax.set_ylim(ax.get_ylim()[0], 0.2)

axes[1, 1].set_xlabel("lag (s)")
axes[0, 0].set_ylabel("encoding perf (r ± sem)")
fig.savefig("../figures/figure3/lags-rois.svg")
plt.show()